In [1]:
# @title Install dependencies
! pip install -q \
  torch \
  transformers \
  datasets \
  huggingface_hub \
  pandas \
  numpy \
  tqdm \
  regex \
  hf-transfer


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
# @title Deterministic Run
import random
import numpy as np
import torch
import os

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.enabled = False

In [3]:
import torch
import pandas as pd

from tqdm import tqdm
torch.set_grad_enabled(False)
tqdm.pandas()

def set_hs_patch_hooks(model, hs_patch_config):
    def patch_hs(name, position_hs):
        def hook(module, input, output):
            # output is a tuple, typically (hidden_states,)
            # hidden_states shape: (batch_size, sequence_length, hidden_size)
            for position_, hs_ in position_hs:
                # Apply hs_ to all items in the batch at position_
                # Assuming hs_ is (hidden_state_dim,) and needs to be broadcast to (batch_size, 1, hidden_state_dim)
                # output[0] is (batch_size, sequence_length, hidden_size)
                output[0][:, position_, :] = hs_.unsqueeze(0)

        return hook

    hooks = []
    for layer in hs_patch_config:
        hooks.append(model.model.layers[layer].register_forward_hook(
            patch_hs(f"patch_hs_{layer}", hs_patch_config[layer])
        ))

    return hooks

def remove_hooks(hooks):
    for hook in hooks:
        hook.remove()

def generate_greedy_deterministic(model, tokenizer, hs_patch_config, batched_input, max_length, end_token):
    """Generates text greedily for a batch of inputs."""
    input_ids = batched_input["input_ids"].to(model.device)
    attention_mask = batched_input["attention_mask"].to(model.device)
    batch_size = input_ids.shape[0]

    # Tracks which sequences have finished generating to stop earlier if all are done
    finished_sequences = torch.zeros(batch_size, dtype=torch.bool, device=model.device)

    with torch.no_grad():
        for _ in range(max_length):
            if finished_sequences.all():
                break

            # Ensure attention_mask matches current input_ids length for model input
            current_sequence_length = input_ids.shape[1]
            if attention_mask.shape[1] < current_sequence_length:
                padding_needed = current_sequence_length - attention_mask.shape[1]
                # For sequences that are still active, append 1 to attention_mask
                # For sequences already finished, we technically don't need to append, but for consistent shape we will.
                attention_mask = torch.cat([
                    attention_mask,
                    torch.ones(batch_size, padding_needed, dtype=torch.long, device=model.device)
                ], dim=1)

            if hs_patch_config is None:
                outputs = model(input_ids, attention_mask=attention_mask, output_attentions=True, output_hidden_states=True)
            else:
                patch_hooks = set_hs_patch_hooks(model, hs_patch_config)
                outputs = model(input_ids, attention_mask=attention_mask, output_attentions=True, output_hidden_states=True)
                remove_hooks(patch_hooks)

            logits = outputs.logits[:, -1, :]
            next_token_ids = torch.argmax(logits, dim=-1) # (batch_size,)

            # For sequences that have finished, ensure they append the end_token or pad_token_id
            # This prevents them from generating meaningful new tokens while others are still generating.
            # Then we can just extend `input_ids` with `next_token_ids` directly.
            new_tokens_to_append = next_token_ids.clone()
            # If a sequence is finished, its next token can be forced to end_token or pad_token
            # This simplifies the concatenation and prevents error if a finished sequence tries to generate further.
            new_tokens_to_append[finished_sequences] = end_token # Or tokenizer.pad_token_id if preferred

            # Append new tokens
            input_ids = torch.cat([input_ids, new_tokens_to_append.unsqueeze(1)], dim=-1)

            # Update attention mask for the newly added tokens. Only active sequences get 1.
            attention_mask = torch.cat([
                attention_mask,
                (~finished_sequences).unsqueeze(1).long() # 1 for active, 0 for finished
            ], dim=-1)

            # Update which sequences are now finished
            current_step_finished = (next_token_ids == end_token)
            finished_sequences = finished_sequences | current_step_finished

    generated_texts = tokenizer.batch_decode(input_ids, skip_special_tokens=True)
    return generated_texts

def decode_tokens(tokenizer, token_array):
  if hasattr(token_array, "shape") and len(token_array.shape) > 1:
    return [decode_tokens(tokenizer, row) for row in token_array]
  return [tokenizer.decode([t]) for t in token_array]

def find_token_range(tokenizer, token_array, substring):
  """Find the tokens corresponding to the given substring in token_array."""
  toks = decode_tokens(tokenizer, token_array)
  whole_string = "".join(toks)
  char_loc = whole_string.index(substring)
  loc = 0
  tok_start, tok_end = None, None
  for i, t in enumerate(toks):
    loc += len(t)
    if tok_start is None and loc > char_loc:
      tok_start = i
    if tok_end is None and loc >= char_loc + len(substring):
      tok_end = i + 1
      break
  return (tok_start, tok_end)

def generate(model, tokenizer, device, df, hs_patch_config=None, batch_size=256):
    """Generates completions for questions in batches."""
    # Drop duplicates and reset index to ensure unique questions are processed once
    df_unique_questions = df[["question"]].drop_duplicates().reset_index(drop=True)
    records = []

    # Iterate through questions in batches
    for i in tqdm(range(0, len(df_unique_questions), batch_size), desc="Generating in batches"):
        batch_df = df_unique_questions.iloc[i:i+batch_size]
        questions_batch = batch_df["question"].tolist()

        # Tokenize the batch, ensuring padding to the longest sequence in the batch and truncation
        batched_input = tokenizer(
            questions_batch,
            return_tensors="pt",
            padding=True, # Pad to the longest sequence in the batch
            truncation=True,
            max_length=tokenizer.model_max_length # Use model_max_length or a suitable maximum
        )

        # Call the modified batched generation function
        deterministic_generations = generate_greedy_deterministic(
            model, tokenizer, hs_patch_config, batched_input, 64, tokenizer.eos_token_id
        )

        # Collect results for the batch
        for q_original, gen_text in zip(questions_batch, deterministic_generations):
            record = {
                "question": q_original,
                "deterministic_generation": gen_text,
            }
            records.append(record)

    df_results = pd.DataFrame(records)
    return df_results

In [4]:
# @title Set up model and tokenizer
import torch
import transformers
import re

def set_requires_grad(requires_grad, *models):
  for model in models:
    if isinstance(model, torch.nn.Module):
      for param in model.parameters():
        param.requires_grad = requires_grad
    elif isinstance(model, (torch.nn.Parameter, torch.Tensor)):
      model.requires_grad = requires_grad
    else:
      assert False, "unknown type %r" % type(model)

class GPTModelAndTokenizer:
  """An object to hold a GPT-style language model and tokenizer."""

  def __init__(
      self,
      model_name=None,
      model=None,
      tokenizer=None,
      low_cpu_mem_usage=False,
      torch_dtype=None,
      ):
    if tokenizer is None:
      assert model_name is not None
      tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
      # Fix: Set pad_token to eos_token if it's not defined
      if tokenizer.pad_token is None:
          tokenizer.pad_token = tokenizer.eos_token

    if model is None:
      assert model_name is not None
      model = transformers.AutoModelForCausalLM.from_pretrained(
          model_name, low_cpu_mem_usage=low_cpu_mem_usage,
          torch_dtype=torch_dtype
          )
      set_requires_grad(False, model)
    self.tokenizer = tokenizer
    self.model = model
    self.layer_names = [
        n
        for n, _ in model.named_modules()
        if (re.match(r"^(transformer|gpt_neox)\.(h|layers)\.\d+$", n))
    ]
    self.num_layers = len(self.layer_names)
    self.vocabulary_projection_function = lambda x, layer: self.model.lm_head(self.model.transformer.ln_f(x)) if layer < self.num_layers else self.model.lm_head(x)
    self.mlp_hidden_size = self.model.config.n_embd * 4
    print(self.mlp_hidden_size)
    print(self.model.config)


In [9]:
# @title Load KEEN PopQA questions
from datasets import load_dataset

keen_popqa_dataset = load_dataset("dhgottesman/keen_estimating_knowledge_in_llms", data_files="popqa_questions.csv")
len(keen_popqa_dataset)

1

In [10]:
# Convert keen_popqa_dataset from huggingface dataset to pandas dataframe
keen_popqa_df = keen_popqa_dataset["train"].to_pandas()
keen_popqa_df.head()

,Unnamed: 0,subj,s_uri,o_uri,prop,obj,question,s_aliases,o_aliases,possible_answers,label
0,0,'71,http://www.wikidata.org/entity/Q12100227,http://www.wikidata.org/entity/Q1174756,composer,David Holmes,Who was the composer of '71?,"['Seventy One', 'Seventy-one']",[],['David Holmes'],head
1,1,'71,http://www.wikidata.org/entity/Q12100227,http://www.wikidata.org/entity/Q130232,genre,drama film,What genre is '71?,"['Seventy One', 'Seventy-one']",['drama movie'],"['crime film', 'thriller movie', 'film action'...",head
2,2,'71,http://www.wikidata.org/entity/Q12100227,http://www.wikidata.org/entity/Q145,country of origin,United Kingdom,What is the country of origin of '71?,"['Seventy One', 'Seventy-one']","['GBR', 'The UK', 'The United Kingdom of Great...","['GBR', 'The UK', 'The United Kingdom of Great...",head
3,3,'71,http://www.wikidata.org/entity/Q12100227,http://www.wikidata.org/entity/Q15715283,director,Yann Demange,Who was the director of '71?,"['Seventy One', 'Seventy-one']",[],['Yann Demange'],head
4,4,'71,http://www.wikidata.org/entity/Q12100227,http://www.wikidata.org/entity/Q22006653,color,color,What color is '71?,"['Seventy One', 'Seventy-one']","['color film', 'colour', 'full color', 'colour...","['colour film', 'color film', 'color', 'colour...",head


In [7]:
# Start with gpt2 for testing that code works
# Subsequently used gpt2-xl
# @title Load gpt2-xl model and tokenizer

gpt_model = GPTModelAndTokenizer(model_name="gpt2-xl")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

6400
GPT2Config {
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 1600,
  "n_head": 25,
  "n_inner": null,
  "n_layer": 48,
  "n_positions": 1024,
  "output_past": true,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.57.3",
  "use_cache": true,
  "vocab_size": 50257
}



In [11]:
# @title Generate completions for questions

# Get just about 10 samples from keen_popqa_df
# keen_popqa_df = keen_popqa_df.sample(10)

# Check for whether is it cuda or cpu
# params to generate function(model, tokenizer, device, df, hs_patch_config=None)
device = "cuda" if torch.cuda.is_available() else "cpu"
gpt_model.model.to(device) # Move the model to the correct device
keen_answer_df = generate(gpt_model.model, gpt_model.tokenizer, device, keen_popqa_df)

Generating in batches: 100%|██████████| 76/76 [3:32:01<00:00, 167.39s/it]  


In [12]:
len(keen_answer_df)

19203

In [13]:
# @title Let us push generated dataset to HF

from huggingface_hub import notebook_login
notebook_login()

In [14]:
from datasets import Dataset, DatasetDict

ds = Dataset.from_pandas(keen_answer_df, preserve_index=False)
dataset_dict = DatasetDict({"train": ds})

In [15]:
repo_id = "kokolamba/keen_popqa_gpt2xl_generations"

dataset_dict.push_to_hub(repo_id, private=False)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/kokolamba/keen_popqa_gpt2xl_generations/commit/87eb3001b659122aaf4736b29b1952f971c6b5b4', commit_message='Upload dataset', commit_description='', oid='87eb3001b659122aaf4736b29b1952f971c6b5b4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/kokolamba/keen_popqa_gpt2xl_generations', endpoint='https://huggingface.co', repo_type='dataset', repo_id='kokolamba/keen_popqa_gpt2xl_generations'), pr_revision=None, pr_num=None)